# 7.2 Trees and Boosting

**Machine learning series, class 2.** In **7.1**, a logistic regression predicted new type 2 diabetes within 2 years with a test AUC of about **0.74**, and LASSO, with bent lines and interactions, did no better. Is that the limit of what enrollment data can tell us, or did the models miss something? In this class we try two of the most popular machine learning methods for tables of data, **random forests** and **gradient boosting**, on the **same people** and the **same test set**, and look inside them.

### By the end of this notebook you'll be able to

1. Explain how a **decision tree**, a **random forest**, and **gradient boosting** make predictions
2. Fit a random forest (**ranger**) and a boosted model (**xgboost**) with tidymodels
3. **Tune** boosting with cross-validation, and read the results
4. Compare models fairly: the same test set, with **AUC**, **Brier score**, and **calibration**
5. Look inside any model with **permutation importance** and **partial dependence**

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Do flexible tree-based models predict new type 2 diabetes within 2 years better than logistic regression, from the same enrollment data? |
| 🧪 **The analysis** | The 7.1 data and the same 75/25 split; logistic regression (7.1's model 3), a random forest, and gradient boosting tuned by 3-fold cross-validation; AUC, Brier score, and calibration on the test set; permutation importance; partial dependence for BMI and age |
| 🎯 **Why** | Trees find curves and interactions by themselves. If they don't beat logistic regression, the signal in these predictors is simple |
| ✅ **What we conclude** | Whether flexibility helps here, which predictors each model relies on, and what shape of risk each model learned for BMI and age |
| ⚠️ **What it can't tell us** | Cause. Importance means "the model uses it", not "it causes diabetes" |
| 🔗 **Connections** | Everything in **7.1**; logistic regression from **3.2** |

| | |
|---|---|
| 📂 **Uses** | `ml_t2d_2yr.csv`, saved by **7.1** (run 7.1 first) |
| 💾 **Saves** | nothing |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: trees, forests, boosting | 20 |
| 2. Setup, the 7.1 data, and the same split | 5 |
| 3. A random forest | 10 |
| 4. Gradient boosting, tuned | 15 |
| 5. A fair comparison: AUC, Brier score, calibration | 15 |
| 6. Inside the models: importance and partial dependence | 20 |
| Wrap-up | 10 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 95 minutes |
| ⏱️ **Run time** | about **15 minutes** (boosting's tuning is the slow part), plus up to **10 minutes the first time** to install ranger and xgboost |
| 💰 **Data cost** | **\$0**: no queries; it reads the file 7.1 saved |
| 💰 **App cost** | about **\$0.27 an hour** while the app is running. The app's disk costs about **\$0.70 a day**, even while the app is stopped |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### A decision tree

A **decision tree** is a flowchart of yes/no questions. Each question splits people into two groups that differ as much as possible in risk; at the end of each path (a **leaf**), the predicted risk is the share of people in that leaf who were diagnosed.

```
                      BMI under 30?
                 yes /             \ no
        general health            age under 50?
        Excellent/Very Good?     yes /      \ no
         yes /     \ no          risk 4%    risk 11%
        risk 1%   risk 4%
```

(Made-up numbers, to show the idea.) Trees find **curves** (several splits on BMI) and **interactions** (the age split only happens for people with a higher BMI) **by themselves**: in **7.1**, we had to add splines and interactions by hand.

But a single deep tree **overfits**: change a few people, and the splits change. The two methods in this class fix that by combining **many** trees.

| | Random forest | Gradient boosting |
|---|---|---|
| **Idea** | many deep trees, each grown on a **random sample** of people, choosing from a **random subset** of predictors at each split; their predictions are **averaged** | many **small** trees built **one after another**; each new tree corrects the **mistakes** of the trees so far |
| **Like** | asking a crowd and averaging their answers | a student who studies their wrong answers, again and again |
| **Main settings** | number of trees; smallest leaf size | number of trees; **learning rate** (how much each tree corrects); **tree depth** |
| **Tuning** | works well with defaults | needs tuning: too many trees, or too fast a learning rate, overfits |
| **R package** | `ranger` | `xgboost` |

### A fair comparison

Every model is trained on the **same training set** and scored on the **same test set** as in 7.1 (the same seed gives the same split). Settings are chosen by **cross-validation** on the training set only.

| Measure | Asks | Better is |
|---|---|---|
| **AUC** | does the model **rank** future cases above others? | higher (0.5 = coin toss) |
| **Brier score** | how far is each predicted risk from what happened (1 or 0), on average, squared? Checks ranking **and** the numbers at once | lower |
| **Calibration** | in each tenth of predicted risk, does the average prediction match the share diagnosed? | points on the diagonal |

## Part 2. Setup, the 7.1 data, and the same split

The setup cell installs **ranger** and **xgboost** the first time (up to 10 minutes). Then we read the file **7.1** saved, and set the **factor levels** again: a CSV file stores text, not the order of the groups, so we restore it (and the reference groups) ourselves.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

ML_PACKAGES <- c("rsample", "recipes", "parsnip", "workflows", "tune", "dials", "yardstick", "ranger", "xgboost")
install_if_missing(ML_PACKAGES)   # first time only
suppressPackageStartupMessages(for (pkg in ML_PACKAGES) library(pkg, character.only = TRUE))

CORES <- parallel::detectCores()   # the app's processors: ranger and xgboost use them all

PREDICTORS <- c("age", "sex_at_birth", "bmi", "sbp", "general_health", "income", "education", "smoking_status")

ml_data <- load_data("ml_t2d_2yr.csv", col_types = cols(person_id = "c")) %>%
  mutate(income         = str_remove_all(income, "\\$"),          # files saved before November 2026 had $ signs
         t2d_future     = factor(t2d_future, levels = c("no", "yes")),
         sex_at_birth   = factor(sex_at_birth),
         general_health = factor(general_health, levels = c("Excellent", "Very Good", "Good", "Fair", "Poor", "Missing")),
         income         = factor(income, levels = c("100k or more", "50k to 100k", "25k to 50k", "Under 25k",
                                                    "Prefer not to answer", "Missing")),
         education      = factor(education, levels = c("College degree", "Some college", "High school or GED",
                                                       "Less than high school", "Missing")),
         smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current", "Missing")))

# Check: a group name that doesn't match the levels above would silently become NA
ml_data %>% summarise(across(all_of(PREDICTORS), ~ sum(is.na(.x))))

**🔍 Read the output:** the number of missing values in each predictor. **Every one should be 0**: 7.1 kept only people with BMI and blood pressure, and turned missing survey answers into a **Missing** group. A number above 0 means a group name in the file doesn't match the levels in this cell. Check it with, for example, `count(ml_data, income)`.

### The same split as 7.1

The same data, in the same order, with the same seed, give the **same** training and test sets.

In [ ]:
model_data <- ml_data %>% select(t2d_future, all_of(PREDICTORS))

set.seed(2026)
split <- initial_split(model_data, prop = 0.75, strata = t2d_future)
train <- training(split)
test  <- testing(split)

bind_rows(training = count(train, t2d_future), test = count(test, t2d_future), .id = "set")

**🔍 Read the output:** the same numbers as in 7.1's Part 5. If they differ, the file has changed since (for example, 7.1 was rerun with `HORIZON_YEARS <- 3`): rerun 7.1 with 2 years.

### The baseline: 7.1's model 3

We refit 7.1's best logistic regression, so all three models can be compared side by side.

In [ ]:
logistic_fit <- workflow() %>%
  add_formula(t2d_future ~ .) %>%
  add_model(logistic_reg()) %>%
  fit(data = train)

test_auc <- function(fit) {
  roc_auc_vec(test$t2d_future, predict(fit, test, type = "prob")$.pred_yes, event_level = "second")
}

round(test_auc(logistic_fit), 3)

**🔍 Read the output:** the test AUC of 7.1's model 3 again (0.743 in our test run).

`test_auc()` is a shorter version of 7.1's: `predict(type = "prob")` gives each test person's predicted risk, `.pred_yes`, and `roc_auc_vec()` turns those risks and the true outcomes into one number.

## Part 3. A random forest

`rand_forest()` with the **ranger** engine. Two settings:

| Setting | Our value | Why |
|---|---|---|
| `trees` | 500 | more trees never overfit a forest; they only take longer. 500 is plenty for 8 predictors |
| `min_n` | 50 | the smallest group a tree may split. With a rare outcome, tiny leaves give jumpy risks (0% or 100%): 50 keeps them steadier |

Trees don't need dummy variables or scaling: ranger splits groups directly.

In [ ]:
rf_fit <- workflow() %>%
  add_formula(t2d_future ~ .) %>%
  add_model(rand_forest(trees = 500, min_n = 50) %>%
              set_engine("ranger", num.threads = CORES, seed = 2026) %>%
              set_mode("classification")) %>%
  fit(data = train)

round(test_auc(rf_fit), 3)

**🔍 Read the output:** the random forest's test AUC. Compare it with logistic regression's, above.

## Part 4. Gradient boosting, tuned

Boosting has settings that **must** be tuned. We try 6 combinations, and score each by **3-fold cross-validation** on the training set (3 folds instead of 5, to save time):

| Setting | Values | Means |
|---|---|---|
| `tree_depth` | 2, 4, 6 | how many questions each small tree may ask. Depth 2 can capture two-way interactions |
| `learn_rate` | 0.02, 0.1 | how much each new tree corrects. Smaller is slower but steadier |
| `trees` | 500 (fixed) | how many trees are added, one after another |

xgboost needs **numbers**, so the recipe turns groups into 0/1 columns (`step_dummy()`). ⏱️ This is the slow cell: about 5 to 10 minutes.

In [ ]:
xgb_workflow <- workflow() %>%
  add_recipe(recipe(t2d_future ~ ., data = train) %>% step_dummy(all_nominal_predictors())) %>%
  add_model(boost_tree(trees = 500, tree_depth = tune(), learn_rate = tune(), min_n = 50) %>%
              set_engine("xgboost", nthread = CORES) %>%
              set_mode("classification"))

set.seed(2026)
folds <- vfold_cv(train, v = 3, strata = t2d_future)

xgb_tuning <- tune_grid(xgb_workflow,
                        resamples = folds,
                        grid      = crossing(tree_depth = c(2, 4, 6), learn_rate = c(0.02, 0.1)),
                        metrics   = metric_set(roc_auc))

collect_metrics(xgb_tuning) %>%
  transmute(tree_depth, learn_rate, cv_auc = round(mean, 4), std_err = round(std_err, 4)) %>%
  arrange(desc(cv_auc))

**🔍 Read the output:** the 6 combinations, best first, with their **cross-validated** AUC and its standard error. Look for:

- **Small differences.** If the top few are within one or two standard errors of each other, any of them is fine.
- **Deep trees and a fast learning rate** (depth 6, rate 0.1) can be **worse**: 500 trees that each correct a lot, with many questions each, start fitting noise. That's **overfitting**, caught by cross-validation, without touching the test set.

We keep the best combination, refit it on the whole training set, and score it on the test set.

In [ ]:
xgb_fit <- xgb_workflow %>%
  finalize_workflow(select_best(xgb_tuning, metric = "roc_auc")) %>%
  fit(data = train)

round(test_auc(xgb_fit), 3)

**🔍 Read the output:** boosting's test AUC.

## Part 5. A fair comparison: AUC, Brier score, calibration

`score()` measures any model on the test set. The **Brier score** is the average of (predicted risk − what happened)², where what happened is 1 or 0. With about 5.5% new cases, a model that gave **everyone** 5.5% would score about 0.052: a useful model must do better (lower).

In [ ]:
fits <- list("Logistic regression" = logistic_fit, "Random forest" = rf_fit, "Gradient boosting" = xgb_fit)

score <- function(fit, model) {
  risk <- predict(fit, test, type = "prob")$.pred_yes
  tibble(model,
         test_auc = round(roc_auc_vec(test$t2d_future, risk, event_level = "second"), 3),
         brier    = round(mean((as.integer(test$t2d_future == "yes") - risk)^2), 4))   # (risk - what happened)², averaged
}

bind_rows(imap(fits, score) %>% list_rbind(),
          tibble(model    = "Everyone gets the average risk",
                 test_auc = 0.5,
                 brier    = round(mean((as.integer(test$t2d_future == "yes") - mean(train$t2d_future == "yes"))^2), 4)))

**🔍 Read the output:** one row per model, and a "no model" row for comparison. Differences in AUC under about **0.01**, or in Brier score in the fourth decimal place, are within the noise of a test set this size.

If the three models are about equal, that **is** the finding: with these 8 predictors, the signal is mostly straight lines (BMI and age up, better self-rated health down), and logistic regression already captures it. Flexible models pay off when there are complex interactions, or many predictors.

### Calibration, side by side

The 7.1 calibration table, for all three models. Each point is a tenth of the test set (about 3,800 people). Tenths with 1 to 20 new cases are hidden, plus the next smallest if only one would be (it could be worked out from the total).

In [ ]:
calibration <- function(fit, model) {
  tibble(risk = predict(fit, test, type = "prob")$.pred_yes, t2d = test$t2d_future == "yes") %>%
    mutate(risk_group = ntile(risk, 10)) %>%
    group_by(risk_group) %>%
    summarise(people = n(), predicted_pct = 100 * mean(risk), n = sum(t2d)) %>%
    hide_small_counts() %>%
    mutate(n = if (sum(is.na(n)) == 1) replace(n, which.min(n), NA) else n,
           observed_pct = 100 * n / people,
           model = model)
}

calibration_all <- imap(fits, calibration) %>% list_rbind()

calibration_all %>%
  filter(!is.na(observed_pct)) %>%
  ggplot(aes(x = predicted_pct, y = observed_pct, colour = model)) +
  geom_abline(linetype = "dashed", colour = "grey50") +
  geom_line() +
  geom_point(size = 2.5) +
  labs(x = "Predicted risk (%)", y = "Diagnosed within 2 years (%)", colour = NULL,
       title = "Calibration on the test set", subtitle = "One point per tenth of predicted risk") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom")

**🔍 Read the output:** points on the dashed line mean the risks are right. Logistic regression is usually well calibrated. Random forests often **squeeze** risks towards the middle (too low at the top, too high at the bottom), because each leaf averages many people; boosting is usually close to the line. The lowest-risk tenths may be missing: they had too few new cases to show.

## Part 6. Inside the models

Trees don't have odds ratios. Two methods work for **any** model, and let us compare all three the same way.

### Permutation importance: which predictors does each model rely on?

Take one predictor in the test set, **shuffle** it (so it no longer matches the person), and see how much the AUC drops. A big drop means the model relied on it. Shuffling keeps the predictor's overall distribution, but breaks its link with everything else.

In [ ]:
permutation_importance <- function(fit, model) {
  base_auc <- test_auc(fit)
  map(PREDICTORS, function(predictor) {
    shuffled <- test %>% mutate("{predictor}" := sample(.data[[predictor]]))
    risk     <- predict(fit, shuffled, type = "prob")$.pred_yes
    tibble(model, predictor,
           auc_drop = base_auc - roc_auc_vec(shuffled$t2d_future, risk, event_level = "second"))
  }) %>% list_rbind()
}

set.seed(2026)
importance <- imap(fits, permutation_importance) %>% list_rbind()

importance %>%
  mutate(predictor = fct_reorder(predictor, auc_drop, .fun = mean)) %>%
  ggplot(aes(x = auc_drop, y = predictor, fill = model)) +
  geom_col(position = "dodge") +
  labs(x = "Drop in test AUC when the predictor is shuffled", y = NULL, fill = NULL,
       title = "Permutation importance", subtitle = "Bigger = the model relies on it more") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom")

**🔍 Read the output:** one bar per predictor and model. Expect **BMI** first by far, then **general health** and **age**. Predictors with bars near 0 add little once the others are known. Do the three models agree? If they rely on the same predictors, they've found the same signal.

**Importance isn't cause.** General health is important because people who feel unwell may already have undiagnosed diabetes, not because rating your health "Excellent" prevents it.

### Partial dependence: what shape did each model learn?

Set **everyone's** BMI to 20, and average their predicted risks; then 22, 24, … up to 50. The line shows how predicted risk changes with BMI, **holding everything else as it is**. Logistic regression can only draw one smooth S-shaped curve; trees can draw any shape.

We use 5,000 randomly chosen test-set people, to keep it quick. Each point is an **average over all 5,000**, so no point describes one person.

In [ ]:
set.seed(2026)
pd_people <- slice_sample(test, n = 5000)

partial_dependence <- function(fit, model, predictor, values) {
  map(values, function(value) {
    tibble(model, predictor, value,
           mean_risk_pct = 100 * mean(predict(fit, mutate(pd_people, "{predictor}" := value), type = "prob")$.pred_yes))
  }) %>% list_rbind()
}

pd_all <- bind_rows(
  imap(fits, ~ partial_dependence(.x, .y, "bmi", seq(20, 50, by = 2))) %>% list_rbind(),
  imap(fits, ~ partial_dependence(.x, .y, "age", seq(20, 80, by = 5))) %>% list_rbind())

pd_all %>%
  mutate(predictor = recode(predictor, bmi = "BMI (kg/m²)", age = "Age at enrollment (years)")) %>%
  ggplot(aes(x = value, y = mean_risk_pct, colour = model)) +
  geom_line(linewidth = 1) +
  facet_wrap(~ predictor, scales = "free_x") +
  labs(x = NULL, y = "Average predicted risk (%)", colour = NULL,
       title = "Partial dependence", subtitle = "Everyone's value set to x; risks averaged") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom")

**🔍 Read the output:** two panels, BMI and age, one line per model.

- **Logistic regression** draws a smooth curve that keeps rising (on the log-odds scale, it's a straight line).
- **The trees** draw **steps**, and often **flatten** at the ends, where there are few people to learn from. Where they bend away from the logistic curve, they've found a shape logistic regression couldn't. If the lines are close, the straight-line assumption was fine.
- Look at **age**: risk may rise in middle age and level off, or even fall, in older people. A logistic regression with plain age can't show that; a spline (7.1's LASSO) or a tree can.

## Which method, when?

Each method has a job it does best. No method wins everywhere, so the honest approach is to try a simple one first and keep a complex one only if it earns its place.

| | Logistic regression | LASSO | Random forest | Gradient boosting |
|---|---|---|---|---|
| **Best when** | a few predictors, chosen in advance, and the model must be explained | **many** predictors for the number of cases (thousands of proteins in 7.3) | you want a flexible model that works well without much tuning | many people, curves and interactions, and time to tune |
| **Finds curves and interactions** | only those you add (splines, interaction terms) | only those you add | by itself | by itself |
| **Explaining it** | easy: one odds ratio per predictor | easy: the predictors it kept, and their weights | hard: needs importance and partial dependence | hard: the same |
| **Tuning** | none | one setting (the penalty) | little: the defaults usually work | several settings, by cross-validation |
| **Watch out for** | a number's effect that bends, if you didn't add a spline | the predictors it picks can change from one sample to another | risks pulled away from 0% and 100%: check calibration; uses a lot of memory | overfitting, if the learning rate is too fast or there are too many trees |

**A practical order:**

1. **Start with logistic regression.** It's the baseline: quick, and easy to explain.
2. **Many predictors?** Try **LASSO**.
3. **Then try a flexible model.** Gradient boosting is often the stronger of the two tree methods on tables like ours.
4. **Keep the complex model only if** it's clearly better on the **same test set**, in AUC **and** calibration, by enough to matter to the people who'll use it. If not, the simple model wins: it's easier to explain, check, and trust.

## Your turn

1. **Overfit on purpose.** Refit the random forest with `min_n = 2` (tiny leaves). What happens to its test AUC and Brier score? Why does the Brier score suffer more than the AUC?
2. **Another predictor.** Which predictor would you add to these models, if it were **known at enrollment**? How would you check it isn't leakage?
3. **Python.** Open `7.2b trees_in_python.ipynb` and fit gradient boosting with **scikit-learn**, on the same file. Do you get a similar AUC?

## 💬 Discuss

1. If all three models score about the same, which would you choose for a clinic, and why? (Think about explaining a risk to a patient, and checking the model for errors.)
2. A model that's more accurate for the **average** person can be less accurate for a small group. How would you check? (**7.4**.)
3. Trees learned that risk flattens at very high BMI. Is that biology, or too few people in that range to learn from?

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *ml_t2d_2yr.csv not found* (or similar) | 7.1 hasn't been run in this workspace | run 7.1 to the end: its last cell saves the file |
| The missing-value check shows numbers above 0 | a group name in the file doesn't match the levels in the setup cell | `count(ml_data, income)` (or the predictor shown), then fix the levels |
| The split counts differ from 7.1's | the file changed, e.g. 7.1 was rerun with a 3-year window | rerun 7.1 with `HORIZON_YEARS <- 2` |
| *there is no package called 'xgboost'* | the install didn't finish | rerun the setup cell; it can take 10 minutes the first time |
| `tune_grid()` is slow | 6 combinations × 3 folds × 500 trees | normal: 5 to 10 minutes |
| The app runs out of memory | forests on 100,000+ people use several GB | stop other notebooks' kernels, or use a larger app |

## 🧑‍🏫 Instructor notes

- **Draw a tree** on the board with the class, from 3 or 4 questions: students should see that trees find interactions by themselves.
- **The punchline may be "no better"**: that's a real and common finding for tables with a few strong predictors. A systematic review of 71 studies found no performance benefit of machine learning over logistic regression, in comparisons at low risk of bias (Christodoulou et al., *Journal of Clinical Epidemiology*, 2019, [doi:10.1016/j.jclinepi.2019.02.004](https://doi.org/10.1016/j.jclinepi.2019.02.004)). Ask why.
- **Permutation importance vs odds ratios:** both say BMI matters most, in different units. Neither is causal.
- **Partial dependence is an average**, not one person's curve: a safe thing to plot from *All of Us* data, unlike one person's predictions.

## Key takeaways

- **Random forests** average many deep trees; **gradient boosting** adds many small trees, each fixing the last one's mistakes.
- Boosting needs **tuning**; cross-validation does it without touching the test set.
- Compare models on the **same test set**, with **AUC** (ranking), **Brier score**, and **calibration** (the numbers).
- **Permutation importance** and **partial dependence** look inside any model, and are safe to share as summaries.
- More flexible isn't always better: with a few strong predictors, logistic regression is hard to beat.

## Clean up

**Clear outputs** before committing or sharing: **Edit → Clear Outputs of All Cells**, then save.

**Next:** **7.3 protein_signature**: thousands of proteins and a few hundred new cases. Here, LASSO earns its keep.